# Tutorial for Sigma parser

In [2]:
import sys
import os

# Add the parent directory to path to import qfoil
sys.path.append(os.path.dirname(os.getcwd()))

from qfoil import QFOIL

_pure_qfoil = QFOIL()

def parse_and_evaluate(expression, context):
    return _pure_qfoil.evaluate(expression, bindings=context)

### Load a PyJuice model for probabilistic queries

In [3]:
import torch
from pathlib import Path

from qfoil import QFOIL
from qfoil.backends.pyjuice import PyJuiceModel, PyJuiceProbabilityBackend

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = PyJuiceModel.load(str(Path(os.getcwd()) / "models" / "mnist_binary_small_0_vs_1"), device=device)

# Wire a parser to an evaluator backed by this model, so VAL/COND/EP/CEP work.
# parse_and_evaluate() (used above and below) never needs a model - it builds a
# plain QFOIL() each call, which is enough for everything except these
# probabilistic queries.
parser = QFOIL(
    probabilistic_backend=PyJuiceProbabilityBackend(model)
)


def evaluate(expr, context):
    return parser.evaluator.evaluate(parser.parse(expr), context)


model.metadata

Compiling 14 TensorCircuit layers...


100%|██████████| 14/14 [00:02<00:00,  6.26it/s]


Compiling 12 TensorCircuit layers...


100%|██████████| 12/12 [00:00<00:00, 59.38it/s]

✅ Model loaded from /home/alexander/Documents/Doctorado/ProyectodeTesis/qfoil-parser-evaluator/examples/models/mnist_binary_small_0_vs_1
   - 2 class models


{'img_size': 14,
 'num_features': 196,
 'num_samples': 12665,
 'num_samples_class_0': 5923,
 'num_samples_class_1': 6742,
 'model_naming': 'mnist_binary_model'}

In [4]:
# Evaluate some simple expressions
context = {
    "A": 2,
    "B": 3,
    "C": 4
}

In [5]:
expr = "A + B * C"
parse_and_evaluate(expr, context)

14

In [6]:
expr = "A >= B"
parse_and_evaluate(expr, context)


0

In [7]:
expr = "(A + B) <= C * 2"  # 5 <= 8
parse_and_evaluate(expr, context)

1

### Evaluate some tuples operations

In [8]:
context = {
    "A": (1, 0, 1),
    "B": (1, 1, 0),
    "C": (1, 0, 1)
}

In [9]:
expr = "A == C"  # True
print(parse_and_evaluate(expr, context))
expr = "A == B"  # False
print(parse_and_evaluate(expr, context))
expr = "NOT (A == B)"
print(parse_and_evaluate(expr, context))

1
0
1


In [10]:
context = {
    "X": (1, 0, 0, None, 1),
    "Y": (1, None, 0, None, 1),
    "Z": (1, 0, 1, None, 1)
}

In [11]:
expr = "(Y <<< X)"  # True
print(parse_and_evaluate(expr, context))
expr = "(Y <<< Z)"  # False
print(parse_and_evaluate(expr, context))

1
0


In [12]:
expr = "(Y <<< X) * (0.5 + NOT (X == Z))"  # 1.5
print(parse_and_evaluate(expr, context))

1.5


In [13]:
expr = "SIGMA y. (X <<< y)"
print(parse_and_evaluate(expr, context))
expr = "SIGMA y. ((X <<< y) * NOT (X == y))"
print(parse_and_evaluate(expr, context))

Inconclusive(reason='bounded_search_incomplete')
Inconclusive(reason='bounded_search_incomplete')


### Some definitions of queries in Sigma logic and comparison with polinomical counterpart

In [14]:
# FULL EXAMPLE
expr = "SIGMA y. ((A <<< y) * NOT (A == y)) == 0"
context = {"A": (1, 0, None)}  # A is not full
result = parse_and_evaluate(expr, context)
print(result)  # Should print 0

expr_p = "FULL(A)"
result_p = parse_and_evaluate(expr_p, context)
print(result_p)  # Should print 0

Inconclusive(reason='bounded_search_incomplete')
0


In [15]:
# MEET EXAMPLE
# z is a meet of x and y
expr = ("(z <<< x) * (z <<< y) * (SIGMA w. (w <<< x) * (w <<< y) * "
        "(w <<< z) >= SIGMA w. (w <<< x) * (w <<< y))")
context = {"x": (1, 0, None), "y": (1, None, 1), "z": (1, None, None)}
result = parse_and_evaluate(expr, context)
print(result)  # Should print 1

expr_p = "MEET(x, y, z)"
result_p = parse_and_evaluate(expr_p, context)
print(result_p)  # Should print 1

Inconclusive(reason='bounded_search_incomplete')
1


In [16]:
# JOIN EXAMPLE
# z is a join of x and y
expr = ("(x <<< z) * (y <<< z) * (SIGMA w. (x <<< w) * "
        "(y <<< w) * (z <<< w) >= SIGMA w. (x <<< w) * (y <<< w))")
context = {"x": (1, 0, None), "y": (1, None, 1), "z": (1, 0, 1)}
result = parse_and_evaluate(expr, context)
print(result)  # Should print 1

expr_p = "JOIN(x, y, z)"
result_p = parse_and_evaluate(expr_p, context)
print(result_p)  # Should print 1

Inconclusive(reason='bounded_search_incomplete')
1


In [17]:
# PARTITION EXAMPLE
# x and y must mention different attributes
expr = ("(SIGMA z. (L_FULL(z) * (x <<< z) * (y <<< z)) >= 1) * "
        "(SIGMA z. (L_L0(z) * L_MEET(x, y, z)))")
context = [{"x": (1, 0, None), "y": (None, None, 1)},
           {"x": (None, None, 0), "y": (None, None, 1)},
           {"x": (None, None, None), "y": (None, None, 1)}]
for ctx in context:
    result = parse_and_evaluate(expr, ctx)
    print(result, end=' ')  # Should print 1, 0, 1
print()

expr_p = "PARTITION(x, y)"
for ctx in context:
    result_p = parse_and_evaluate(expr_p, ctx)
    print(result_p, end=' ')


Inconclusive(reason='bounded_search_incomplete') Inconclusive(reason='bounded_search_incomplete') Inconclusive(reason='bounded_search_incomplete') 
1 0 1 

In [18]:
expr = "L0(x)"
context = {"x": (1, None, None)}
result = parse_and_evaluate(expr, context)
print(result)  # Should print 0

0


In [19]:
# L1 EXAMPLE
expr = "SIGMA y. (L_L0(y) * ((y <<< x) * !(y == x)) * (SIGMA z.((y <<< z) * !(y == z) * (z <<< x) * !(z == x)) == 0 ))"
context = [{"x": (1, 0, None)},
           {"x": (None, None, 0)},
           {"x": (None, None, None)}]
for ctx in context:
    result = parse_and_evaluate(expr, ctx)
    print(result, end=' ')
print()
expr = "L1(x)"
for ctx in context:
    result = parse_and_evaluate(expr, ctx)
    print(result, end=' ')

Inconclusive(reason='bounded_search_incomplete') Inconclusive(reason='bounded_search_incomplete') Inconclusive(reason='bounded_search_incomplete') 
0 1 0 

In [20]:
expr = "L1(x) * L1(y) * (SIGMA z.(x <<< z) * (y <<< z) == 0)"
context = [{"x": (1, 0, 0, None, 1), "y": (1, None, 0, None, 1)},
           {"x": (None, None, 0, None, None), "y": (None, None, 1, None, None)},
           {"x": (None, None, None, None, 1), "y": (None, None, None, None, 1)}]
for ctx in context:
    result = parse_and_evaluate(expr, ctx)
    print(result, end=' ')
print()
expr = "OPPOSITE(x, y)"
for ctx in context:
    result = parse_and_evaluate(expr, ctx)
    print(result, end=' ')

0 Inconclusive(reason='bounded_search_incomplete') Inconclusive(reason='bounded_search_incomplete') 
0 1 0 

In [21]:
# Count the number of combinations of L1 tuples that are opposites = 2(n+1)
expr = "SIGMA x. (SIGMA y. (OPPOSITE(x, y) ))"
context = {"dummy": (1,1,1,1,1)}
print(parse_and_evaluate(expr, context))


Inconclusive(reason='bounded_search_incomplete')


In [22]:
# Same undefined features (SUF) example
cond1 = "((u <<< x) * (u <<< y) * !(v <<< x) * !(v <<< y))"
cond2 = "((v <<< x) * (v <<< y) * !(u <<< x) * !(u <<< y))"
cond3 = "((u <<< x) * (v <<< y) * !(u <<< y) * !(v <<< x))"
cond4 = "((u <<< y) * (v <<< x) * !(u <<< x) * !(v <<< y))"
cond5 = "(!(u <<< x) * !(u <<< y) * !(v <<< x) * !(v <<< y))"

count_opposite = "SIGMA u. SIGMA v. (OPPOSITE(u, v))"

expr = f"SIGMA u. SIGMA v. (OPPOSITE(u, v) * ({cond1} + {cond2} + {cond3} + {cond4} + {cond5})) == {count_opposite}"

context = [{"x": (1, None, None, None), "y": (0, 1, None, None)},
           {"x": (1, 0, None, None), "y": (0, 1, None, None)},
           {"x": (1, None, None, 1), "y": (1, None, None, 1)},
           {"x": (None, None, 1, None), "y": (None, None, 0, None)}]
for ctx in context:
      result = parse_and_evaluate(expr, ctx)
      print(result, end=' ')
print()
expr = "SUF(x, y)"
for ctx in context:
      result = parse_and_evaluate(expr, ctx)
      print(result, end=' ')


Inconclusive(reason='bounded_search_incomplete') Inconclusive(reason='bounded_search_incomplete') Inconclusive(reason='bounded_search_incomplete') Inconclusive(reason='bounded_search_incomplete') 
0 1 1 1 

### Basic inference queries to the model

These use the PyJuice model loaded above: 196 binary pixel features per instance (`None` = unobserved). The class label is **not** part of the tuple - it's a separate argument to `VAL`/`COND`/`EP`/`CEP` (class 0 or class 1 for this model).

In [23]:
# VAL EXAMPLE
# VAL(x) = Pr(x), marginalized over the class variable
# VAL(x, c) = Pr(x, Y=c), the joint probability of x and class c
context = [
    {"x": tuple([0] * 196)},        # evidence: all-black image
    {"x": tuple([1] * 196)},        # evidence: all-white image
    {"x": tuple([None] * 196)},     # no evidence at all - should return ~1
]
for ctx in context:
    marginal = evaluate("VAL(x)", ctx)
    joint_0 = evaluate("VAL(x, 0)", ctx)
    joint_1 = evaluate("VAL(x, 1)", ctx)
    print(f"Pr(x)={marginal}, Pr(x,Y=0)={joint_0}, Pr(x,Y=1)={joint_1}")

Pr(x)=2.9765057861705746e-05, Pr(x,Y=0)=1.3594145592759973e-12, Pr(x,Y=1)=2.9765057861705746e-05
Pr(x)=5.007968296712159e-113, Pr(x,Y=0)=5.007968296712159e-113, Pr(x,Y=1)=7.567720773820281e-119
Pr(x)=0.9999998807907176, Pr(x,Y=0)=0.4676667380960311, Pr(x,Y=1)=0.532333113343457


In [24]:
# COND EXAMPLE
# COND(x, y, c) = Pr(x, Y=c | y), for disjoint partial instances x, y
expr = "COND(x, y, c)"
query = tuple([1] + [None] * 195)  # x: only the first pixel is on
context = [
    {"x": query, "y": tuple([0, 1] + [None] * 194), "c": 1},  # partial evidence y
    {"x": query, "y": tuple([None] * 196), "c": 1},            # no evidence: Pr(x, Y=1)
    {"x": query, "y": tuple([0] + [None] * 195), "c": 1},       # y conflicts with x -> 0
]
for ctx in context:
    result = evaluate(expr, ctx)
    print(f"Pr(x, Y={ctx['c']} | y): {result}")

Pr(x, Y=1 | y): 0.0
Pr(x, Y=1 | y): 0.00016901484835424088
Pr(x, Y=1 | y): 0.0


In [25]:
# EP / CEP EXAMPLE
# EP(x, c) := COND(BOTTOM, x, c) = Pr(Y=c | x) - the expected prediction of
# class c given the (possibly partial) evidence x.
# CEP(x, c) = 1 - EP(x, c) = Pr(Y != c | x)
x = tuple([1, 0, 1] + [None] * 193)  # only the first 3 pixels observed
context = {"x": x, "c": 1}
ep = evaluate("EP(x, c)", context)
cep = evaluate("CEP(x, c)", context)
print(f"Pr(Y=1 | x) = {ep}")
print(f"Pr(Y!=1 | x) = {cep}  (should equal 1 - EP)")

Pr(Y=1 | x) = 0.5840752518144804
Pr(Y!=1 | x) = 0.4159247481855196  (should equal 1 - EP)


In [26]:
# Free GPU memory held by the PyJuice circuits (only meaningful on CUDA)
if torch.cuda.is_available():
    del model
    torch.cuda.empty_cache()